In [1]:
import tarfile
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import precision_recall_fscore_support

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
import os

cifar_path = '/content/cifar-10-python.tar.gz'

if os.path.exists(cifar_path):
    with tarfile.open(cifar_path, 'r:gz') as tar:
        tar.extractall(path='/content/')
    print("CIFAR-10 extraction complete!")
else:
    print("CIFAR-10 archive not found:", cifar_path)

/tmp/ipykernel_6182/3525889758.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path='/content/')


CIFAR-10 extraction complete!


In [3]:
def load_cifar_batch(file_path):
    with open(file_path, 'rb') as fo:
        dict_data = pickle.load(fo, encoding='bytes')

    images = dict_data[b'data'].reshape(-1, 3, 32, 32)
    images = images.transpose(0, 2, 3, 1)

    labels = dict_data[b'labels']

    return images, labels


class CIFAR10Dataset(Dataset):

    def __init__(self, file_paths, transform=None):

        self.images = []
        self.labels = []

        for path in file_paths:
            imgs, lbls = load_cifar_batch(path)

            self.images.append(imgs)
            self.labels.extend(lbls)

        self.images = np.vstack(self.images)
        self.labels = np.array(self.labels)

        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

In [4]:
train_batches = [
    f'/content/cifar-10-batches-py/data_batch_{i}'
    for i in range(1, 6)
]

test_batch = [
    '/content/cifar-10-batches-py/test_batch'
]

# Simple transform for checking the dataset
basic_transform = transforms.ToTensor()

train_dataset_check = CIFAR10Dataset(
    train_batches,
    transform=basic_transform
)

test_dataset_check = CIFAR10Dataset(
    test_batch,
    transform=basic_transform
)

print("Training images:", len(train_dataset_check))
print("Test images:", len(test_dataset_check))

Training images: 50000
Test images: 10000


In [5]:
class FlexibleCNN(nn.Module):

    def __init__(
        self,
        num_layers=4,
        use_bn=False,
        dropout_rate=0.0
    ):

        super(FlexibleCNN, self).__init__()

        self.layers = nn.ModuleList()

        in_channels = 3

        if num_layers == 4:
            features = [32, 64, 128, 256]

        elif num_layers == 5:
            features = [32, 64, 128, 256, 512]

        else:
            raise ValueError("Only 4 or 5 layers are supported.")

        for out_channels in features:

            block = []

            block.append(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=3,
                    stride=1,
                    padding=1
                )
            )

            if use_bn:
                block.append(
                    nn.BatchNorm2d(out_channels)
                )

            block.append(nn.ReLU())

            block.append(
                nn.MaxPool2d(
                    kernel_size=2,
                    stride=2
                )
            )

            self.layers.append(
                nn.Sequential(*block)
            )

            in_channels = out_channels

        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Dropout
        if dropout_rate > 0:
            self.dropout = nn.Dropout(dropout_rate)
        else:
            self.dropout = nn.Identity()

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(in_channels, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10)
        )

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        x = self.gap(x)

        x = self.dropout(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [6]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print("Using device:", device)

model_test = FlexibleCNN(
    num_layers=4,
    use_bn=False,
    dropout_rate=0.0
)

model_test = model_test.to(device)

dummy_input = torch.randn(
    4, 3, 32, 32
).to(device)

output = model_test(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)

Using device: cuda
Input shape: torch.Size([4, 3, 32, 32])
Output shape: torch.Size([4, 10])


In [7]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=5,
    device='cuda'
):

    if isinstance(device, str):
        device = torch.device(device)

    model = model.to(device)

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_acc': []
    }

    for epoch in range(num_epochs):

        model.train()

        running_loss = 0.0
        total_samples = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item() * images.size(0)
            )

            total_samples += images.size(0)

        train_loss = (
            running_loss / total_samples
        )

        val_loss, val_acc, _, _, _ = evaluate_model(
            model,
            val_loader,
            criterion,
            device
        )

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Val Acc: {val_acc:.2f}%"
        )

    return history


def evaluate_model(
    model,
    loader,
    criterion,
    device='cuda'
):

    if isinstance(device, str):
        device = torch.device(device)

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    total_samples = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() * images.size(0)
            )

            total_samples += images.size(0)

            _, predicted = torch.max(
                outputs,
                1
            )

            all_preds.extend(
                predicted.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    val_loss = (
        running_loss / total_samples
    )

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    accuracy = (
        all_preds == all_labels
    ).mean() * 100

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_preds,
            average='macro',
            zero_division=0
        )
    )

    return (
        val_loss,
        accuracy,
        precision,
        recall,
        f1
    )

In [8]:
transform_train = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])


train_dataset = CIFAR10Dataset(
    train_batches,
    transform=transform_train
)

test_dataset = CIFAR10Dataset(
    test_batch,
    transform=transform_test
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=100,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


criterion = nn.CrossEntropyLoss()


print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 782
Testing batches: 100


In [9]:
test_model = FlexibleCNN(
    num_layers=4,
    use_bn=False,
    dropout_rate=0.0
).to(device)

test_optimizer = optim.Adam(
    test_model.parameters(),
    lr=0.001
)

print("Adam optimizer created successfully!")

Adam optimizer created successfully!


In [10]:
print("\n--- Training Baseline 4-Layer CNN ---")

baseline_model = FlexibleCNN(
    num_layers=4,
    use_bn=False,
    dropout_rate=0.0
)

optimizer_baseline = optim.Adam(
    baseline_model.parameters(),
    lr=0.001
)

baseline_history = train_model(
    baseline_model,
    train_loader,
    test_loader,
    criterion,
    optimizer_baseline,
    num_epochs=5,
    device=device
)

(
    _,
    b_acc,
    b_prec,
    b_rec,
    b_f1
) = evaluate_model(
    baseline_model,
    test_loader,
    criterion,
    device
)


--- Training Baseline 4-Layer CNN ---
Epoch [1/5] | Train Loss: 1.4539 | Val Loss: 1.1737 | Val Acc: 57.67%
Epoch [2/5] | Train Loss: 0.9490 | Val Loss: 0.8731 | Val Acc: 69.45%
Epoch [3/5] | Train Loss: 0.7376 | Val Loss: 0.7605 | Val Acc: 73.63%
Epoch [4/5] | Train Loss: 0.6044 | Val Loss: 0.7771 | Val Acc: 73.65%
Epoch [5/5] | Train Loss: 0.5004 | Val Loss: 0.7606 | Val Acc: 75.44%


In [11]:
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


b_params = count_parameters(
    baseline_model
)

print("Baseline parameters:", b_params)

Baseline parameters: 661834


In [12]:
print("\n--- Training Tuned 4-Layer CNN ---")

tuned_model = FlexibleCNN(
    num_layers=4,
    use_bn=True,
    dropout_rate=0.3
)

optimizer_tuned = optim.Adam(
    tuned_model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

tuned_history = train_model(
    tuned_model,
    train_loader,
    test_loader,
    criterion,
    optimizer_tuned,
    num_epochs=5,
    device=device
)

(
    _,
    t_acc,
    t_prec,
    t_rec,
    t_f1
) = evaluate_model(
    tuned_model,
    test_loader,
    criterion,
    device
)

t_params = count_parameters(
    tuned_model
)


--- Training Tuned 4-Layer CNN ---
Epoch [1/5] | Train Loss: 1.2282 | Val Loss: 0.9540 | Val Acc: 66.09%
Epoch [2/5] | Train Loss: 0.8996 | Val Loss: 0.8340 | Val Acc: 70.69%
Epoch [3/5] | Train Loss: 0.7655 | Val Loss: 0.7807 | Val Acc: 72.84%
Epoch [4/5] | Train Loss: 0.6803 | Val Loss: 0.7174 | Val Acc: 74.97%
Epoch [5/5] | Train Loss: 0.6059 | Val Loss: 0.7334 | Val Acc: 74.10%


In [13]:
resnet_train_transform = transforms.Compose([
    transforms.ToPILImage(),

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])


resnet_test_transform = transforms.Compose([
    transforms.ToPILImage(),

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])


resnet_train_dataset = CIFAR10Dataset(
    train_batches,
    transform=resnet_train_transform
)

resnet_test_dataset = CIFAR10Dataset(
    test_batch,
    transform=resnet_test_transform
)


resnet_train_loader = DataLoader(
    resnet_train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

resnet_test_loader = DataLoader(
    resnet_test_dataset,
    batch_size=100,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("ResNet data loaders ready!")

ResNet data loaders ready!


In [ ]:
print("\n--- Training Pre-trained ResNet-18 ---")

pretrained_resnet = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace ImageNet's 1000-class output
# with CIFAR-10's 10 classes

pretrained_resnet.fc = nn.Linear(
    pretrained_resnet.fc.in_features,
    10
)

pretrained_resnet = pretrained_resnet.to(device)


optimizer_resnet = optim.Adam(
    pretrained_resnet.parameters(),
    lr=0.0001
)


resnet_history = train_model(
    pretrained_resnet,
    resnet_train_loader,
    resnet_test_loader,
    criterion,
    optimizer_resnet,
    num_epochs=5,
    device=device
)


(
    _,
    r_acc,
    r_prec,
    r_rec,
    r_f1
) = evaluate_model(
    pretrained_resnet,
    resnet_test_loader,
    criterion,
    device
)


r_params = count_parameters(
    pretrained_resnet
)


--- Training Pre-trained ResNet-18 ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 130MB/s]


Epoch [1/5] | Train Loss: 0.3214 | Val Loss: 0.1913 | Val Acc: 93.68%
Epoch [2/5] | Train Loss: 0.0972 | Val Loss: 0.1877 | Val Acc: 94.02%


In [ ]:
print("\n========== FINAL RESULTS ==========")

print("\nBaseline 4-Layer CNN")
print("Accuracy :", f"{b_acc:.2f}%")
print("Precision:", f"{b_prec:.4f}")
print("Recall   :", f"{b_rec:.4f}")
print("F1 Score :", f"{b_f1:.4f}")
print("Parameters:", f"{b_params:,}")


print("\nTuned 4-Layer CNN")
print("Accuracy :", f"{t_acc:.2f}%")
print("Precision:", f"{t_prec:.4f}")
print("Recall   :", f"{t_rec:.4f}")
print("F1 Score :", f"{t_f1:.4f}")
print("Parameters:", f"{t_params:,}")



In [ ]:

print("\nPretrained ResNet-18")
print("Accuracy :", f"{r_acc:.2f}%")
print("Precision:", f"{r_prec:.4f}")
print("Recall   :", f"{r_rec:.4f}")
print("F1 Score :", f"{r_f1:.4f}")
print("Parameters:", f"{r_params:,}")